# Choice Set Collection and Metric Evaluation           
                                                     
Notebook for:                                                                                                                             
1. Generate OD pairs                                                                                                                    
2. Collect choice sets via ORS and GraphHopper                                                                                            
3. Evaluate choice set quality using performance metrics

In [ ]:
import json
import time
import requests
import numpy as np
import folium
from folium import plugins
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import polyline as pl
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import LineString
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ORS_KEY = "" # if you have an ORS API key, put it here.
GH_KEY = ""
TARGET_COUNT = 10

ORS_DELAY = 1.6  # seconds between ORS requests                                                                                             
GH_DELAY = 1.2   # seconds between GH requests 

## Load stations and generate OD pairs   

In [ ]:
estacoes = pd.read_parquet("estacoes.parquet")
print(f"Stations: {len(estacoes)}")
estacoes.head()

In [ ]:
def gerar_pares_od(estacoes, max_dist_km=10, min_dist_km=0.5):
    """Generate OD pairs between stations filtered by straight-line distance.

    Parameters
    ----------
    estacoes : DataFrame
        Stations with columns id, name, lat, lon.
    max_dist_km : float
        Maximum distance between pairs in km (default 10).
    min_dist_km : float
        Minimum distance between pairs in km (default 0.5).

    Returns
    -------
    DataFrame
        OD pairs with origin/destination columns.
    """
    gdf = gpd.GeoDataFrame(
        estacoes,
        geometry=gpd.points_from_xy(estacoes['lon'], estacoes['lat']),
        crs='EPSG:4326'
    ).to_crs(epsg=31983)  # UTM zone 23S for distance in metres

    pairs = []
    ids = estacoes['id'].values
    names = estacoes['name'].values
    lats = estacoes['lat'].values
    lons = estacoes['lon'].values
    geoms = gdf.geometry.values

    for i in range(len(estacoes)):
        for j in range(i + 1, len(estacoes)):
            dist_m = geoms[i].distance(geoms[j])
            dist_km = dist_m / 1000

            if min_dist_km <= dist_km <= max_dist_km:
                pairs.append({
                    'od_id': f"OD_{ids[i]}_{ids[j]}",
                    'od_name': f"{names[i]} -> {names[j]}",
                    'origin_id': ids[i],
                    'origin_name': names[i],
                    'origin_lat': lats[i],
                    'origin_lon': lons[i],
                    'destination_id': ids[j],
                    'destination_name': names[j],
                    'destination_lat': lats[j],
                    'destination_lon': lons[j],
                    'dist_reta_km': round(dist_km, 2),
                    'coletada_ors': False,
                    'coletada_gh': False,
                })

    df = pd.DataFrame(pairs)
    print(f"Pairs generated: {len(df)}")
    print(f"Mean distance: {df['dist_reta_km'].mean():.1f} km")
    print(f"Min distance: {df['dist_reta_km'].min():.1f} km, Max: {df['dist_reta_km'].max():.1f} km")
    return df


rotas_exemplo = gerar_pares_od(estacoes, max_dist_km=10, min_dist_km=0.5)
rotas_exemplo.to_parquet("rotas_exemplo.parquet", index=False)
rotas_exemplo.head(10)

## Collection Functions

In [ ]:
def _aggregate_extra_info(extra_block):
    """Aggregate the percentage distribution of an ORS extra_info block.

    Parameters
    ----------
    extra_block : dict
        Block with 'values' and 'summary' keys.

    Returns
    -------
    dict
        {value_id: percentage}.
    """
    summary = extra_block.get("summary", [])
    return {item["value"]: item["amount"] for item in summary}


def _aggregate_gh_detail(detail_entries):
    """Aggregate GraphHopper path_details into a per-segment distribution.

    Parameters
    ----------
    detail_entries : list
        List of [from_idx, to_idx, value] triples.

    Returns
    -------
    dict
        {value: segment_count}.
    """
    dist = {}
    for entry in detail_entries:
        if len(entry) == 3:
            val = str(entry[2])
            span = entry[1] - entry[0]
            dist[val] = dist.get(val, 0) + span
    return dist

In [ ]:
def fetch_ors_routes(origin, destination, api_key, target_count=3):
    """Fetch alternative cycling routes via the ORS Directions API (cycling-regular).

    Parameters
    ----------
    origin : list
        [lon, lat].
    destination : list
        [lon, lat].
    api_key : str
        ORS API key.
    target_count : int
        Number of alternative routes requested (default 3).

    Returns
    -------
    list
        List of dicts with route attributes.
    """
    url = "https://api.openrouteservice.org/v2/directions/cycling-regular/json"
    headers = {"Authorization": api_key, "Content-Type": "application/json"}

    all_routes = []
    seen_distances = set()
    preferences = ["recommended", "shortest", "fastest"]

    for pref in preferences:
        body = {
            "coordinates": [origin, destination],
            "alternative_routes": {
                "target_count": min(target_count, 3),
                "weight_factor": 1.8,
                "share_factor": 0.6,
            },
            "attributes": ["avgspeed", "detourfactor", "percentage"],
            "elevation": True,
            "extra_info": ["steepness", "suitability", "surface", "waycategory", "waytype"],
            "geometry": True,
            "instructions": True,
            "units": "m",
            "preference": pref,
        }

        resp = requests.post(url, json=body, headers=headers)
        if resp.status_code == 429:
            print("    ORS rate limit, waiting 60s...")
            time.sleep(60)
            resp = requests.post(url, json=body, headers=headers)
        if resp.status_code != 200:
            print(f"    ORS {resp.status_code}: {resp.text[:200]}")
            continue

        data = resp.json()

        for route in data.get("routes", []):
            summary = route.get("summary", {})
            dist_key = round(summary.get("distance", 0), 1)
            if dist_key in seen_distances:
                continue
            seen_distances.add(dist_key)

            segments = route.get("segments", [{}])
            extras = route.get("extras", {})
            geometry = route.get("geometry", "")
            coords = []
            if geometry:
                try:
                    coords = pl.decode(geometry)
                except Exception:
                    coords = [(0, 0)]

            n_steps = sum(len(seg.get("steps", [])) for seg in segments)

            all_routes.append({
                "route_index": len(all_routes),
                "preference": pref,
                "distance_m": summary.get("distance", 0),
                "duration_s": summary.get("duration", 0),
                "ascent_m": summary.get("ascent", 0),
                "descent_m": summary.get("descent", 0),
                "n_instructions": n_steps,
                "n_waypoints": len(coords),
                "bbox": str(route.get("bbox", [])),
                "surface_distribution": json.dumps(_aggregate_extra_info(extras.get("surface", {}))),
                "waytype_distribution": json.dumps(_aggregate_extra_info(extras.get("waytype", {}))),
                "steepness_distribution": json.dumps(_aggregate_extra_info(extras.get("steepness", {}))),
                "suitability_distribution": json.dumps(_aggregate_extra_info(extras.get("suitability", {}))),
                "waycategory_distribution": json.dumps(_aggregate_extra_info(extras.get("waycategory", {}))),
                "geometry_encoded": geometry,
            })

        time.sleep(ORS_DELAY)
        if len(all_routes) >= target_count:
            break

    return all_routes[:target_count]

In [ ]:
def fetch_gh_routes(origin, destination, api_key, max_paths=3):
    """Fetch alternative cycling routes via the GraphHopper Routing API (bike).

    Parameters
    ----------
    origin : list
        [lon, lat].
    destination : list
        [lon, lat].
    api_key : str
        GraphHopper API key.
    max_paths : int
        Maximum number of alternative routes (default 3).

    Returns
    -------
    list
        List of dicts with route attributes.
    """
    url = "https://graphhopper.com/api/1/route"
    params = {"key": api_key}
    body = {
        "profile": "bike",
        "points": [origin, destination],
        "points_encoded": False,
        "elevation": True,
        "instructions": True,
        "calc_points": True,
        "details": [
            "road_class", "road_environment", "surface",
            "smoothness", "max_speed", "average_speed",
            "distance", "time",
        ],
        "algorithm": "alternative_route",
        "alternative_route.max_paths": max_paths,
        "alternative_route.max_weight_factor": 2.0,
        "alternative_route.max_share_factor": 0.6,
    }

    resp = requests.post(url, json=body, params=params)
    if resp.status_code == 429:
        print("    GH rate limit, waiting 60s...")
        time.sleep(60)
        resp = requests.post(url, json=body, params=params)
    resp.raise_for_status()
    data = resp.json()

    routes = []
    for i, path in enumerate(data.get("paths", [])):
        pts = path.get("points", {})
        coords = pts.get("coordinates", []) if isinstance(pts, dict) else []
        details = path.get("details", {})

        routes.append({
            "route_index": i,
            "distance_m": path.get("distance", 0),
            "duration_ms": path.get("time", 0),
            "duration_s": path.get("time", 0) / 1000.0,
            "ascend_m": path.get("ascend", 0),
            "descend_m": path.get("descend", 0),
            "n_instructions": len(path.get("instructions", [])),
            "n_waypoints": len(coords),
            "bbox": str(path.get("bbox", [])),
            "weight": path.get("weight", 0),
            "transfers": path.get("transfers", 0),
            "points_order": str(path.get("points_order", [])),
            "road_class_distribution": json.dumps(_aggregate_gh_detail(details.get("road_class", []))),
            "surface_distribution": json.dumps(_aggregate_gh_detail(details.get("surface", []))),
            "smoothness_distribution": json.dumps(_aggregate_gh_detail(details.get("smoothness", []))),
            "road_environment_distribution": json.dumps(_aggregate_gh_detail(details.get("road_environment", []))),
            "geometry_wkt": json.dumps(coords),
        })

    return routes

In [ ]:
def build_choice_set(od_row, routes, source):
    """Build choice-set records for a single OD pair.

    Parameters
    ----------
    od_row : Series
        Row from the OD pairs DataFrame.
    routes : list
        List of route dicts.
    source : str
        'ors' or 'gh'.

    Returns
    -------
    list
        List of dicts, one per route.
    """
    if not routes:
        return []

    rows = []
    for route in routes:
        row = {
            "od_id": od_row["od_id"],
            "od_name": od_row["od_name"],
            "origin_lon": od_row["origin_lon"],
            "origin_lat": od_row["origin_lat"],
            "destination_lon": od_row["destination_lon"],
            "destination_lat": od_row["destination_lat"],
            "source": source,
            "choice_set_size": len(routes),
            "chosen": 1 if route["route_index"] == 0 else 0,
        }
        row.update(route)
        rows.append(row)
    return rows

## Choice Set Collection

Runs collection for all OD pairs. If the rate limit is hit, waits and resumes.

In [ ]:
# load progress if it exists
progress_path = Path("rotas_exemplo.parquet")
if progress_path.exists():
    rotas_exemplo = pd.read_parquet(progress_path)

ors_csv = Path("choiceset_ors.csv")
gh_csv = Path("choiceset_graphhopper.csv")

# load previous collections
if ors_csv.exists():
    df_ors_prev = pd.read_csv(ors_csv)
    ors_rows = df_ors_prev.to_dict('records')
    coletados_ors = set(df_ors_prev['od_id'].unique())
    rotas_exemplo.loc[rotas_exemplo['od_id'].isin(coletados_ors), 'coletada_ors'] = True
else:
    ors_rows = []
    coletados_ors = set()

if gh_csv.exists():
    df_gh_prev = pd.read_csv(gh_csv)
    gh_rows = df_gh_prev.to_dict('records')
    coletados_gh = set(df_gh_prev['od_id'].unique())
    rotas_exemplo.loc[rotas_exemplo['od_id'].isin(coletados_gh), 'coletada_gh'] = True
else:
    gh_rows = []
    coletados_gh = set()

pendentes = rotas_exemplo[~(rotas_exemplo['coletada_ors'] & rotas_exemplo['coletada_gh'])]
print(f"Pending pairs: {len(pendentes)}")
print(f"Already collected — ORS: {len(coletados_ors)} | GH: {len(coletados_gh)}")

In [ ]:
def coletar_rotas(rotas_exemplo, ors_rows, gh_rows, coletados_ors, coletados_gh):
    """Collect ORS and GH routes for pending OD pairs, saving after each pair.

    Parameters
    ----------
    rotas_exemplo : DataFrame
        OD pairs DataFrame with coletada_ors/coletada_gh flags.
    ors_rows : list
        Already collected ORS route dicts.
    gh_rows : list
        Already collected GH route dicts.
    coletados_ors : set
        Set of od_ids already collected by ORS.
    coletados_gh : set
        Set of od_ids already collected by GH.
    """
    pendentes = rotas_exemplo[~(rotas_exemplo['coletada_ors'] & rotas_exemplo['coletada_gh'])]
    total = len(pendentes)

    for n, (idx, od) in enumerate(pendentes.iterrows(), 1):
        od_id = od['od_id']
        origin = [od['origin_lon'], od['origin_lat']]
        destination = [od['destination_lon'], od['destination_lat']]
        salvou = False

        print(f"\n[{n}/{total}] {od['od_name']} ({od['dist_reta_km']} km)")

        if od_id not in coletados_ors:
            try:
                ors_routes = fetch_ors_routes(origin, destination, ORS_KEY, target_count=TARGET_COUNT)
                print(f"  ORS: {len(ors_routes)} routes")
                ors_rows.extend(build_choice_set(od, ors_routes, "ors"))
                rotas_exemplo.loc[idx, 'coletada_ors'] = True
                coletados_ors.add(od_id)
                salvou = True
            except Exception as e:
                print(f"  ORS error: {e}")
            time.sleep(ORS_DELAY)

        if od_id not in coletados_gh:
            try:
                gh_routes = fetch_gh_routes(origin, destination, GH_KEY, max_paths=TARGET_COUNT)
                print(f"  GH: {len(gh_routes)} routes")
                gh_rows.extend(build_choice_set(od, gh_routes, "gh"))
                rotas_exemplo.loc[idx, 'coletada_gh'] = True
                coletados_gh.add(od_id)
                salvou = True
            except Exception as e:
                print(f"  GH error: {e}")
            time.sleep(GH_DELAY)

        if salvou:
            pd.DataFrame(ors_rows).to_csv("choiceset_ors.csv", index=False)
            pd.DataFrame(gh_rows).to_csv("choiceset_graphhopper.csv", index=False)
            rotas_exemplo.to_parquet("rotas_exemplo.parquet", index=False)

    print(f"\nCollection complete: ORS={len(ors_rows)} routes, GH={len(gh_rows)} routes")

In [ ]:
coletar_rotas(rotas_exemplo, ors_rows, gh_rows, coletados_ors, coletados_gh)

## Load Results

In [ ]:
df_ors = pd.read_csv("choiceset_ors.csv")
df_gh = pd.read_csv("choiceset_graphhopper.csv")

print(f"ORS: {df_ors.shape}")
print(f"GH:  {df_gh.shape}")

print(f"\nOD pairs with ORS routes: {df_ors['od_id'].nunique()}")
print(f"OD pairs with GH routes: {df_gh['od_id'].nunique()}")

print(f"\nRoutes per OD pair (ORS):")
print(df_ors.groupby('od_id')['route_index'].count().describe())
print(f"\nRoutes per OD pair (GH):")
print(df_gh.groupby('od_id')['route_index'].count().describe())

## Choice Set Visualisation

In [ ]:
def plot_choice_set_gradient(df_ors, df_gh, od_id, title=None, plot_chosen=True, plot_rand=True):
    """
    Visualizes a choice set of routes for a given OD pair from ORS and GraphHopper.

    This function displays the routes for the specified OD pair (`od_id`) from both ORS and GraphHopper data, 
    using color gradients based on route distances. A random route is also highlighted for comparison.

    Parameters
    ----------
    df_ors : pd.DataFrame
        DataFrame with ORS route data containing columns like 'od_id', 'route_index', and 'geometry'.
    df_gh : pd.DataFrame
        DataFrame with GraphHopper route data containing columns like 'od_id', 'route_index', and 'geometry'.
    od_id : int
        The OD pair ID to visualize the routes for.
    title : str, optional
        The title of the map (default is None).
    plot_chosen : bool, optional
        Whether to plot the chosen route (default is True).
    plot_rand : bool, optional
        Whether to plot a random route from the choice set (default is True).

    Returns
    -------
    folium.Map
        A Folium map displaying the routes and their comparison.
    """
    # Filter the DataFrames to get the relevant data for the specified OD pair (od_id)
    ors_routes = df_ors[df_ors['od_id'] == od_id]
    gh_routes = df_gh[df_gh['od_id'] == od_id]

    # Create a map centered around the first route's coordinates
    m = folium.Map(location=[ors_routes.iloc[0]['geometry'].y, ors_routes.iloc[0]['geometry'].x], zoom_start=13)

    # Set color scale based on the route distance (gradient)
    min_distance = min(ors_routes['route_length'].min(), gh_routes['route_length'].min())
    max_distance = max(ors_routes['route_length'].max(), gh_routes['route_length'].max())

    # Add the ORS routes to the map with a gradient color scale based on distance
    for _, route in ors_routes.iterrows():
        color = folium.ColorLine(
            route['geometry'].coords,
            colors=['blue'],
            weight=6,
            opacity=0.7
        ).add_to(m)

    # Add the GraphHopper routes to the map
    for _, route in gh_routes.iterrows():
        color = folium.ColorLine(
            route['geometry'].coords,
            colors=['green'],
            weight=6,
            opacity=0.7
        ).add_to(m)

    # Optionally plot a random route from the choice set
    if plot_rand:
        random_route_ors = ors_routes.sample(1)
        random_route_gh = gh_routes.sample(1)
        
        folium.PolyLine(
            locations=random_route_ors.iloc[0]['geometry'].coords,
            color="red", weight=3, opacity=1
        ).add_to(m)
        folium.PolyLine(
            locations=random_route_gh.iloc[0]['geometry'].coords,
            color="red", weight=3, opacity=1
        ).add_to(m)

    # Optionally highlight the chosen route
    if plot_chosen:
        chosen_route_ors = ors_routes.iloc[0]
        chosen_route_gh = gh_routes.iloc[0]
        
        folium.PolyLine(
            locations=chosen_route_ors['geometry'].coords,
            color="blue", weight=3, opacity=1
        ).add_to(m)
        folium.PolyLine(
            locations=chosen_route_gh['geometry'].coords,
            color="green", weight=3, opacity=1
        ).add_to(m)

    # Add markers and popup details for each route
    for _, route in ors_routes.iterrows():
        folium.Marker(
            location=[route['geometry'].y, route['geometry'].x],
            popup=f"ORS Route {route['route_index']}: {route['route_length']} km"
        ).add_to(m)
        
    for _, route in gh_routes.iterrows():
        folium.Marker(
            location=[route['geometry'].y, route['geometry'].x],
            popup=f"GH Route {route['route_index']}: {route['route_length']} km"
        ).add_to(m)

    # Add a title to the map
    if title:
        folium.map.Marker(
            location=[m.location[0] + 0.02, m.location[1]],
            icon=folium.DivIcon(html=f'<h3>{title}</h3>')
        ).add_to(m)

    return m


def plot_choice_set_comparison_widget(df_ors, df_gh, od_id):
    """
    Creates an interactive widget to compare ORS and GraphHopper routes for the selected OD pair.

    This function uses ipywidgets to create a dropdown for selecting the OD pair and color for plotting,
    comparing the routes from ORS and GraphHopper for the same OD pair.

    Parameters
    ----------
    df_ors : pd.DataFrame
        DataFrame with ORS route data containing columns like 'od_id', 'route_index', and 'geometry'.
    df_gh : pd.DataFrame
        DataFrame with GraphHopper route data containing columns like 'od_id', 'route_index', and 'geometry'.
    od_id : int
        The OD pair ID to visualize the routes for.

    Returns
    -------
    ipywidgets.Widget
        The interactive widget for selecting routes and plotting them.
    """
    from ipywidgets import interact, widgets

    # Create widgets for user interaction
    od_widget = widgets.Dropdown(
        options=df_ors['od_id'].unique(),
        value=od_id,
        description='OD Pair:',
        style={'description_width': 'initial'}
    )

    color_widget = widgets.ColorPicker(
        value='blue',
        description='Color:',
        style={'description_width': 'initial'}
    )

    # Update function to create map when parameters change
    def update_map(od_id, color):
        m = plot_choice_set_gradient(df_ors, df_gh, od_id, title=f'OD Pair {od_id}', plot_chosen=True, plot_rand=True)
        m.save(f"route_comparison_{od_id}.html")

    # Link widgets to update function
    interact(update_map, od_id=od_widget, color=color_widget)

## Choice Set Metric Evaluation

Metrics implemented:
- **Overlap**: proportion of shared linear extent between generated and observed route
- **Coverage**: share of observations above an overlap threshold
- **Reproduction Rate**: 1 if overlap >= threshold, 0 otherwise
- **Path Size Factor**: spatial diversity of the choice set (0 = redundant, 1 = unique)
- **Precision**: proportion of generated route points that fall on the observed route
- **Recall**: proportion of observed route points captured by the generated route
- **F1 Score**: harmonic mean of precision and recall
- **Detour Factor**: ratio of route distance to straight-line distance
- **Directness**: inverse of the detour factor

In [ ]:
def route_to_linestring(geometry_data, source='ors'):
    """Converts route geometry to a shapely LineString.

    Parameters
    ----------
    geometry_data : str
        Polyline encoded (ORS) or JSON coords (GH).
    source : str
        'ors' or 'gh'.

    Returns
    -------
    LineString
    """
    if source == 'ors':
        coords = decode_polyline_3d(geometry_data)
        return LineString([(c[1], c[0]) for c in coords])  # (lon, lat)
    else:
        raw = json.loads(geometry_data)
        return LineString([(c[0], c[1]) for c in raw])  # already (lon, lat)


def calc_overlap(route_a, route_b, buffer_m=30):
    """Calculates overlap between two routes.

    Parameters
    ----------
    route_a : LineString
        Reference route (observed).
    route_b : LineString
        Generated route.
    buffer_m : float
        Buffer in meters to consider overlap (default 30).

    Returns
    -------
    float
        Overlap percentage (0-1).
    """
    gdf_a = gpd.GeoDataFrame(geometry=[route_a], crs='EPSG:4326').to_crs(epsg=31983)
    gdf_b = gpd.GeoDataFrame(geometry=[route_b], crs='EPSG:4326').to_crs(epsg=31983)

    buffered_a = gdf_a.geometry.iloc[0].buffer(buffer_m)
    intersection = gdf_b.geometry.iloc[0].intersection(buffered_a)

    len_b = gdf_b.geometry.iloc[0].length
    if len_b == 0:
        return 0.0

    return min(intersection.length / len_b, 1.0)


def calc_reproduction_rate(overlap, threshold=0.8):
    """Calculates the reproduction rate.

    Parameters
    ----------
    overlap : float
        Overlap value (0-1).
    threshold : float
        Minimum overlap threshold (default 0.8).

    Returns
    -------
    int
        1 if overlap >= threshold, 0 otherwise.
    """
    return 1 if overlap >= threshold else 0


def calc_path_size_factor(routes_ls):
    """Calculates the Path Size Factor for a choice set.

    Parameters
    ----------
    routes_ls : list of LineString
        List of LineStrings in the choice set.

    Returns
    -------
    list of float
        Path Size Factor for each route.
    """
    if len(routes_ls) <= 1:
        return [1.0]

    gdf = gpd.GeoDataFrame(geometry=routes_ls, crs='EPSG:4326').to_crs(epsg=31983)
    geoms = gdf.geometry.values
    lengths = [g.length for g in geoms]
    buffer_m = 20

    ps_factors = []
    for i, (geom_i, len_i) in enumerate(zip(geoms, lengths)):
        if len_i == 0:
            ps_factors.append(0.0)
            continue

        # Discretize into segments
        n_segments = max(int(len_i / 10), 10)
        fracs = np.linspace(0, 1, n_segments)
        points = [geom_i.interpolate(f, normalized=True) for f in fracs]

        ps_sum = 0
        for pt in points:
            # Count how many routes pass through this point
            n_sharing = sum(
                1 for j, geom_j in enumerate(geoms)
                if geom_j.distance(pt) <= buffer_m
            )
            segment_len = len_i / n_segments
            ps_sum += (segment_len / len_i) * (1.0 / max(n_sharing, 1))

        ps_factors.append(round(ps_sum, 4))

    return ps_factors


def calc_precision_recall_f1(route_observed, route_generated, buffer_m=30):
    """Calculates precision, recall, and F1 score between observed and generated routes.

    Parameters
    ----------
    route_observed : LineString
        Observed route.
    route_generated : LineString
        Generated route.
    buffer_m : float
        Buffer in meters (default 30).

    Returns
    -------
    tuple
        (precision, recall, f1).
    """
    gdf_obs = gpd.GeoDataFrame(geometry=[route_observed], crs='EPSG:4326').to_crs(epsg=31983)
    gdf_gen = gpd.GeoDataFrame(geometry=[route_generated], crs='EPSG:4326').to_crs(epsg=31983)

    obs_geom = gdf_obs.geometry.iloc[0]
    gen_geom = gdf_gen.geometry.iloc[0]

    # Discretize into points
    n_obs = max(int(obs_geom.length / 10), 10)
    n_gen = max(int(gen_geom.length / 10), 10)

    pts_obs = [obs_geom.interpolate(i / n_obs, normalized=True) for i in range(n_obs + 1)]
    pts_gen = [gen_geom.interpolate(i / n_gen, normalized=True) for i in range(n_gen + 1)]

    obs_buffered = obs_geom.buffer(buffer_m)
    gen_buffered = gen_geom.buffer(buffer_m)

    # Precision: generated points within observed buffer
    precision = sum(1 for pt in pts_gen if obs_buffered.contains(pt)) / len(pts_gen)

    # Recall: observed points within generated buffer
    recall = sum(1 for pt in pts_obs if gen_buffered.contains(pt)) / len(pts_obs)

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return round(precision, 4), round(recall, 4), round(f1, 4)


def calc_detour_factor(route_ls, origin, destination):
    """Calculates the detour factor (route/straight ratio).

    Parameters
    ----------
    route_ls : LineString
        Route.
    origin : tuple
        (lon, lat) of the origin.
    destination : tuple
        (lon, lat) of the destination.

    Returns
    -------
    float
        Detour factor.
    """
    gdf_route = gpd.GeoDataFrame(geometry=[route_ls], crs='EPSG:4326').to_crs(epsg=31983)
    gdf_reta = gpd.GeoDataFrame(
        geometry=[LineString([origin, destination])], crs='EPSG:4326'
    ).to_crs(epsg=31983)

    route_len = gdf_route.geometry.iloc[0].length
    reta_len = gdf_reta.geometry.iloc[0].length

    if reta_len == 0:
        return 0.0
    return round(route_len / reta_len, 4)

## Apply Metrics to All Choice Sets

In [ ]:
def evaluate_choice_set(df, source='ors', overlap_threshold=0.8):
    """Evaluates metrics for all choice sets of a given source.

    Parameters
    ----------
    df : DataFrame
        Route dataset (ORS or GH).
    source : str
        'ors' or 'gh'.
    overlap_threshold : float
        Threshold for reproduction rate (default 0.8).

    Returns
    -------
    DataFrame
        Metrics for each route.
    DataFrame
        Aggregated metrics by OD pair.
    """
    # Select the correct geometry column based on the source (ORS or GH)
    geom_col = 'geometry_encoded' if source == 'ors' else 'geometry_wkt'
    results = []

    # Iterate through each OD pair in the dataset
    for od_id, group in df.groupby('od_id'):
        # Get the chosen route (if available) for the OD pair
        chosen_row = group[group['chosen'] == 1]
        if chosen_row.empty:
            chosen_row = group.iloc[[0]]

        # Convert the geometry of the chosen route to a LineString
        chosen_ls = route_to_linestring(chosen_row.iloc[0][geom_col], source)
        origin = (chosen_row.iloc[0]['origin_lon'], chosen_row.iloc[0]['origin_lat'])
        destination = (chosen_row.iloc[0]['destination_lon'], chosen_row.iloc[0]['destination_lat'])

        # Create a list of LineStrings for the routes in the choice set
        routes_ls = [route_to_linestring(row[geom_col], source) for _, row in group.iterrows()]
        # Calculate the Path Size Factor for each route in the choice set
        ps_factors = calc_path_size_factor(routes_ls)

        # Evaluate each route in the choice set
        for i, (_, row) in enumerate(group.iterrows()):
            route_ls = routes_ls[i]

            # If the route is the chosen one, set overlap and precision/recall to 1
            if row['chosen'] == 1:
                overlap = 1.0
                precision, recall, f1 = 1.0, 1.0, 1.0
            else:
                # Otherwise, calculate overlap and precision/recall
                overlap = calc_overlap(chosen_ls, route_ls)
                precision, recall, f1 = calc_precision_recall_f1(chosen_ls, route_ls)

            # Calculate the detour factor (how much longer the route is compared to the straight line)
            detour = calc_detour_factor(route_ls, origin, destination)

            # Store the metrics for this route
            results.append({
                'od_id': od_id,
                'source': source,
                'route_index': row['route_index'],
                'chosen': row['chosen'],
                'distance_m': row['distance_m'],
                'duration_s': row.get('duration_s', row.get('duration_ms', 0) / 1000),
                'overlap': overlap,
                'reproduction_rate': calc_reproduction_rate(overlap, overlap_threshold),
                'path_size_factor': ps_factors[i],
                'precision': precision,
                'recall': recall,
                'f1_score': f1,
                'detour_factor': detour,
                'directness': round(1 / detour, 4) if detour > 0 else 0,
            })

    # Create a DataFrame of metrics for each route
    df_metrics = pd.DataFrame(results)

    # Aggregate metrics by OD pair
    df_agg = df_metrics.groupby('od_id').agg(
        n_rotas=('route_index', 'count'),
        mean_overlap=('overlap', 'mean'),
        coverage_80=('reproduction_rate', 'mean'),
        mean_ps_factor=('path_size_factor', 'mean'),
        mean_precision=('precision', 'mean'),
        mean_recall=('recall', 'mean'),
        mean_f1=('f1_score', 'mean'),
        mean_detour=('detour_factor', 'mean'),
        dist_min=('distance_m', 'min'),
        dist_max=('distance_m', 'max'),
    ).reset_index()

    return df_metrics, df_agg


# Evaluate ORS metrics
print("Evaluating ORS...")
df_metrics_ors, df_agg_ors = evaluate_choice_set(df_ors, source='ors')
print(f"ORS: {len(df_metrics_ors)} routes evaluated, {len(df_agg_ors)} OD pairs")

# Evaluate GH metrics
print("\nEvaluating GH...")
df_metrics_gh, df_agg_gh = evaluate_choice_set(df_gh, source='gh')
print(f"GH: {len(df_metrics_gh)} routes evaluated, {len(df_agg_gh)} OD pairs")

# Save the results to CSV
df_metrics_ors.to_csv("metrics_ors.csv", index=False)
df_metrics_gh.to_csv("metrics_gh.csv", index=False)
df_agg_ors.to_csv("metrics_agg_ors.csv", index=False)
df_agg_gh.to_csv("metrics_agg_gh.csv", index=False)
print("\nMetrics saved")

## Comparative Summary: ORS vs GraphHopper

In [ ]:
def resumo_comparativo(df_agg_ors, df_agg_gh):
    """Print a comparative summary between ORS and GH.

    Parameters
    ----------
    df_agg_ors : DataFrame
        Aggregated ORS metrics.
    df_agg_gh : DataFrame
        Aggregated GH metrics.
    """
    metrics = ['n_rotas', 'mean_overlap', 'coverage_80', 'mean_ps_factor',
               'mean_precision', 'mean_recall', 'mean_f1', 'mean_detour']

    rows = []
    for m in metrics:
        rows.append({
            'metric': m,
            'ORS_mean': df_agg_ors[m].mean() if m in df_agg_ors.columns else None,
            'ORS_std': df_agg_ors[m].std() if m in df_agg_ors.columns else None,
            'GH_mean': df_agg_gh[m].mean() if m in df_agg_gh.columns else None,
            'GH_std': df_agg_gh[m].std() if m in df_agg_gh.columns else None,
        })

    df_comp = pd.DataFrame(rows)
    print(df_comp.to_string(index=False))
    return df_comp


df_comparativo = resumo_comparativo(df_agg_ors, df_agg_gh)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metrics_to_plot = ['overlap', 'path_size_factor', 'precision', 'recall', 'f1_score', 'detour_factor']

for ax, metric in zip(axes.flat, metrics_to_plot):
    # filter only non-chosen routes for distribution comparison
    ors_vals = df_metrics_ors[df_metrics_ors['chosen'] == 0][metric].dropna()
    gh_vals = df_metrics_gh[df_metrics_gh['chosen'] == 0][metric].dropna()

    ax.hist(ors_vals, bins=30, alpha=0.6, label='ORS', color='steelblue')
    ax.hist(gh_vals, bins=30, alpha=0.6, label='GH', color='coral')
    ax.set_title(metric)
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle('Distribuicao de Metricas - Rotas Alternativas (nao-chosen)', fontsize=14)
plt.tight_layout()
plt.savefig("figs/metricas_distribuicao.png", dpi=150, bbox_inches="tight")
plt.show()